In [1]:
import os
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
)
from src.load_data import download_data, load_file

SEED = 42

In [2]:
path = download_data()
X_train, y_train = load_file(path, "ECG5000_TRAIN.ts")
X_test, y_test = load_file(path, "ECG5000_TEST.ts")


y_train = (y_train != 1).astype(int)
y_test = (y_test != 1).astype(int)
print(X_train.shape, X_test.shape, np.bincount(y_train), np.bincount(y_test))

Path to dataset files: /Users/yugamdeepgoyal/.cache/kagglehub/datasets/salsabilahmid/ecg50000/versions/1
(500, 140) (4500, 140) [292 208] [2627 1873]


In [3]:
param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [2, 3, 4, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.3, 0.5, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "reg_lambda": [0.5, 1, 5, 10],
}

In [4]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1,
)

search = RandomizedSearchCV(
    xgb,
    param_distributions,
    n_iter=40,
    scoring="roc_auc",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    random_state=SEED,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV ROC AUC:", round(search.best_score_, 4))
best = search.best_estimator_

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best params: {'subsample': 0.8, 'reg_lambda': 10, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 0.3}
Best CV ROC AUC: 0.9994


In [9]:
y_prob = best.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

print("\nTesting")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("F1       :", f1_score(y_test, y_pred))
print("ROC AUC  :", roc_auc_score(y_test, y_prob))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Normal", "Abnormal"]))

# Training performance, to check the gap
y_prob_tr = best.predict_proba(X_train)[:, 1]
print("Train accuracy:", accuracy_score(y_train, (y_prob_tr > 0.5).astype(int)))
print("Train Recall:", recall_score(y_train, (y_prob_tr > 0.5).astype(int)))
print("Train Precision:", precision_score(y_train, (y_prob_tr > 0.5).astype(int)))
print("Train F1: ", f1_score(y_train, (y_prob_tr > 0.5).astype(int)))
print("Train Confusion Matrix: \n", confusion_matrix(y_train, (y_prob_tr > 0.5).astype(int)))
print("Train F1: ", f1_score(y_train, (y_prob_tr > 0.5).astype(int)))
print("Train Classification report: \n", classification_report(y_train, (y_prob_tr > 0.5).astype(int)))
print("Train ROC AUC :", roc_auc_score(y_train, y_prob_tr))

os.makedirs("results", exist_ok=True)
np.save("results/y_prob_xgb.npy", y_prob)
np.save("results/y_test.npy", y_test)

os.makedirs("models", exist_ok=True)
joblib.dump(best, "models/xgb_model.pkl")


Testing
Accuracy : 0.972
Recall   : 0.9407367859049653
Precision: 0.9915588069780529
F1       : 0.9654794520547946
ROC AUC  : 0.9973660522753264
Confusion matrix:
 [[2612   15]
 [ 111 1762]]
              precision    recall  f1-score   support

      Normal       0.96      0.99      0.98      2627
    Abnormal       0.99      0.94      0.97      1873

    accuracy                           0.97      4500
   macro avg       0.98      0.97      0.97      4500
weighted avg       0.97      0.97      0.97      4500

Train accuracy: 0.998
Train Recall: 0.9951923076923077
Train Precision: 1.0
Train F1:  0.9975903614457832
Train Confusion Matrix: 
 [[292   0]
 [  1 207]]
Train F1:  0.9975903614457832
Train Classification report: 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       292
           1       1.00      1.00      1.00       208

    accuracy                           1.00       500
   macro avg       1.00      1.00      1.00    

['models/xgb_model.pkl']